# Chapter 31
## ING Rhythms

In [ ]:
import brian2 as b2
import matplotlib.pyplot as plt
import numpy as np


def validate_probability(name, value):
    value = float(value)
    if not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be in [0, 1], got {value}")
    return value


def tau_peak(tau_d_ms, tau_r_ms, tau_dq_ms, dt_ms=0.01):
    s = 0.0
    t = 0.0
    ds = np.exp(-t / tau_dq_ms) * (1.0 - s) / tau_r_ms - s / tau_d_ms
    while ds > 0.0:
        t_old, ds_old = t, ds
        s_mid = s + 0.5 * dt_ms * ds
        ds_mid = (
            np.exp(-(t + 0.5 * dt_ms) / tau_dq_ms)
            * (1.0 - s_mid) / tau_r_ms
            - s_mid / tau_d_ms
        )
        s += dt_ms * ds_mid
        t += dt_ms
        ds = np.exp(-t / tau_dq_ms) * (1.0 - s) / tau_r_ms - s / tau_d_ms
    return (t_old * (-ds) + t * ds_old) / (ds_old - ds)


def solve_tau_dq(tau_d_ms, tau_r_ms, tau_peak_ms):
    left = 1.0
    while tau_peak(tau_d_ms, tau_r_ms, left) > tau_peak_ms:
        left *= 0.5
    right = tau_r_ms
    while tau_peak(tau_d_ms, tau_r_ms, right) < tau_peak_ms:
        right *= 2.0
    while right - left > 1e-12:
        middle = 0.5 * (left + right)
        if tau_peak(tau_d_ms, tau_r_ms, middle) <= tau_peak_ms:
            left = middle
        else:
            right = middle
    return 0.5 * (left + right)


In [ ]:
WB_EQS = """
dv/dt = (g_l*(E_l-v) + g_k*n**4*(E_k-v)
         + g_na*m_inf**3*h*(E_na-v) + i_ext + I_chem + I_gap)/C : volt
dh/dt = (h_inf-h)/tau_h : 1
dn/dt = (n_inf-n)/tau_n : 1
dq/dt = 0.5*(1+tanh(v/(10*mV)))*(1-q)/(0.1*ms) - q/tau_dq : 1
ds/dt = q*(1-s)/tau_r - s/tau_d : 1
m_inf = alpha_m/(alpha_m+beta_m) : 1
h_inf = alpha_h/(alpha_h+beta_h) : 1
n_inf = alpha_n/(alpha_n+beta_n) : 1
alpha_m = 0.1/mV*(v+35*mV)/(1-exp(-(v+35*mV)/(10*mV)))/ms : Hz
beta_m = 4*exp(-(v+60*mV)/(18*mV))/ms : Hz
alpha_h = 0.07*exp(-(v+58*mV)/(20*mV))/ms : Hz
beta_h = 1/(exp(-(v+28*mV)/(10*mV))+1)/ms : Hz
alpha_n = -0.01/mV*(v+34*mV)/(exp(-(v+34*mV)/(10*mV))-1)/ms : Hz
beta_n = 0.125*exp(-(v+44*mV)/(80*mV))/ms : Hz
tau_h = 1/(5*(alpha_h+beta_h)) : second
tau_n = 1/(5*(alpha_n+beta_n)) : second
i_ext : amp
I_chem : amp
I_gap : amp
tau_dq : second
tau_r : second
tau_d : second
C : farad (constant)
g_l : siemens (constant)
g_k : siemens (constant)
g_na : siemens (constant)
E_l : volt (constant)
E_k : volt (constant)
E_na : volt (constant)
"""


def initial_wb_state(count, mode, rng):
    if mode == "uniform_random":
        return {
            "v": rng.uniform(-100.0, 50.0, size=count) * b2.mV,
            "h": rng.uniform(0.0, 1.0, size=count),
            "n": rng.uniform(0.0, 1.0, size=count),
            "q": np.zeros(count),
            "s": np.zeros(count),
        }
    if mode == "fixed":
        return {
            "v": np.full(count, -75.0) * b2.mV,
            "h": np.full(count, 0.1),
            "n": np.full(count, 0.1),
            "q": np.zeros(count),
            "s": np.zeros(count),
        }
    raise ValueError(f"unknown WB initial-state mode: {mode}")


## 1_CELL_ING

In [ ]:
def simulate_single_cell_ing(
    i_ext=1.5 * b2.uA,
    g_ii=0.5 * b2.msiemens,
    tau_d=9 * b2.ms,
    duration=200 * b2.ms,
    dt=0.01 * b2.ms,
    record=True,
):
    b2.start_scope()
    b2.defaultclock.dt = dt

    cells = b2.NeuronGroup(
        1, WB_EQS, method="rk2", threshold="v > -20*mV", refractory="v > -20*mV"
    )
    cells.C = 1.0 * b2.ufarad
    cells.g_l = 0.1 * b2.msiemens
    cells.g_k = 9.0 * b2.msiemens
    cells.g_na = 35.0 * b2.msiemens
    cells.E_l = -65.0 * b2.mV
    cells.E_k = -90.0 * b2.mV
    cells.E_na = 55.0 * b2.mV
    cells.i_ext = i_ext
    cells.tau_dq = solve_tau_dq(float(tau_d / b2.ms), 0.5, 0.5) * b2.ms
    cells.tau_r = 0.5 * b2.ms
    cells.tau_d = tau_d
    cells.v = -75.0 * b2.mV
    cells.h = 0.1
    cells.n = 0.1
    cells.q = 0.0
    cells.s = 0.0

    chem = b2.Synapses(
        cells,
        cells,
        model="g : siemens\nI_chem_post = g*s_pre*(-75*mV-v_post) : amp (summed)",
    )
    chem.connect(i=[0], j=[0])
    chem.g = g_ii

    state_monitor = b2.StateMonitor(cells, "v", record=record)
    spike_monitor = b2.SpikeMonitor(cells)
    b2.run(duration)

    spike_ms = np.asarray(spike_monitor.t / b2.ms)
    if len(spike_ms) < 2:
        raise RuntimeError("fewer than two spikes")
    period_ms = float(spike_ms[-1] - spike_ms[-2])
    if record:
        t_ms = np.asarray(state_monitor.t / b2.ms)
        v_mv = np.asarray(state_monitor.v[0] / b2.mV)
    else:
        t_ms = np.array([])
        v_mv = np.array([])
    return {
        "t_ms": t_ms,
        "v_mv": v_mv,
        "spike_ms": spike_ms,
        "period_ms": period_ms,
        "frequency_hz": 1000.0 / period_ms,
    }


if __name__ == "__main__":
    one_cell = simulate_single_cell_ing()
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot(one_cell["t_ms"], one_cell["v_mv"], color="k", lw=2)
    ax.set(xlabel="time [ms]", ylabel="v [mV]", ylim=(-100, 50))
    fig.tight_layout()


In [ ]:
def compute_ing_condition_numbers(duration=200*b2.ms):
    base = simulate_single_cell_ing(duration=duration, record=False)
    reduced_i = simulate_single_cell_ing(i_ext=1.5*0.99*b2.uA, duration=duration, record=False)
    raised_g = simulate_single_cell_ing(g_ii=0.5*1.01*b2.msiemens, duration=duration, record=False)
    raised_tau = simulate_single_cell_ing(tau_d=9*1.01*b2.ms, duration=duration, record=False)
    period = base["period_ms"]
    return {
        "base_period_ms": period,
        "pct_i_ext": 100.0*(period-reduced_i["period_ms"])/period,
        "pct_g_ii": 100.0*(period-raised_g["period_ms"])/period,
        "pct_tau_d": 100.0*(period-raised_tau["period_ms"])/period,
    }


if __name__ == "__main__":
    condition_numbers = compute_ing_condition_numbers()
    print("base_period", condition_numbers["base_period_ms"])
    print("pct_i_ext", condition_numbers["pct_i_ext"])
    print("pct_g_ii", condition_numbers["pct_g_ii"])
    print("pct_tau_d", condition_numbers["pct_tau_d"])


In [ ]:
ING_CONFIGS = {
    "ING_1": dict(sigma_i=0.00, g_hat_ii=0.5, p_ii=1.00, g_hat_gap=0.00, p_gap=1.00, fixed_indegree=False, init_mode="uniform_random", seed=124875),
    "ING_2": dict(sigma_i=0.03, g_hat_ii=0.5, p_ii=1.00, g_hat_gap=0.00, p_gap=1.00, fixed_indegree=False, init_mode="phase", seed=63806),
    "ING_3": dict(sigma_i=0.00, g_hat_ii=0.5, p_ii=0.85, g_hat_gap=0.00, p_gap=1.00, fixed_indegree=False, init_mode="phase", seed=63806),
    "ING_4": dict(sigma_i=0.00, g_hat_ii=0.5, p_ii=0.85, g_hat_gap=0.00, p_gap=1.00, fixed_indegree=True, init_mode="phase", seed=63806),
    "ING_5": dict(sigma_i=0.05, g_hat_ii=0.5, p_ii=0.50, g_hat_gap=0.00, p_gap=1.00, fixed_indegree=False, init_mode="phase", seed=63806),
    "ING_6": dict(sigma_i=0.05, g_hat_ii=0.5, p_ii=0.50, g_hat_gap=0.10, p_gap=0.05, fixed_indegree=False, init_mode="phase", seed=63806),
    "ING_7": dict(sigma_i=0.00, g_hat_ii=0.5, p_ii=1.00, g_hat_gap=0.00, p_gap=1.00, fixed_indegree=False, init_mode="phase", seed=63806),
    "ING_8": dict(sigma_i=0.00, g_hat_ii=0.5, p_ii=1.00, g_hat_gap=0.00, p_gap=0.05, fixed_indegree=False, init_mode="phase", seed=63806),
    "ING_9": dict(sigma_i=0.05, g_hat_ii=0.5, p_ii=1.00, g_hat_gap=0.00, p_gap=0.05, fixed_indegree=False, init_mode="phase", seed=63806),
    "ING_10": dict(sigma_i=0.05, g_hat_ii=0.5, p_ii=1.00, g_hat_gap=0.04, p_gap=0.05, fixed_indegree=False, init_mode="phase", seed=63806),
}


def _wb_m_inf(v):
    alpha_m = 0.1 * (v + 35.0) / (1.0 - np.exp(-(v + 35.0) / 10.0))
    beta_m = 4.0 * np.exp(-(v + 60.0) / 18.0)
    return alpha_m / (alpha_m + beta_m)


def _wb_h_inf_and_tau(v):
    alpha_h = 0.07 * np.exp(-(v + 58.0) / 20.0)
    beta_h = 1.0 / (np.exp(-(v + 28.0) / 10.0) + 1.0)
    rate = alpha_h + beta_h
    return alpha_h / rate, 1.0 / (5.0 * rate)


def _wb_n_inf_and_tau(v):
    alpha_n = -0.01 * (v + 34.0) / (np.exp(-(v + 34.0) / 10.0) - 1.0)
    beta_n = 0.125 * np.exp(-(v + 44.0) / 80.0)
    rate = alpha_n + beta_n
    return alpha_n / rate, 1.0 / (5.0 * rate)


def wb_phase_initial_state(i_ext_ua, phase, dt_ms=0.01):
    i_ext_ua = np.asarray(i_ext_ua, dtype=float)
    phase = np.asarray(phase, dtype=float)
    if i_ext_ua.ndim != 1 or phase.shape != i_ext_ua.shape:
        raise ValueError("i_ext_ua and phase must be equal-length one-dimensional arrays")
    if i_ext_ua.size == 0:
        raise ValueError("i_ext_ua and phase must not be empty")
    if np.any(~np.isfinite(i_ext_ua)) or np.any(~np.isfinite(phase)):
        raise ValueError("i_ext_ua and phase must contain finite values")
    if np.any((phase < 0.0) | (phase > 1.0)):
        raise ValueError("phase must be in [0, 1]")
    dt_ms = float(dt_ms)
    if not np.isfinite(dt_ms) or dt_ms <= 0.0:
        raise ValueError("dt_ms must be positive")

    count = i_ext_ua.size
    v = np.full(count, -70.0)
    h, _ = _wb_h_inf_and_tau(v)
    n, _ = _wb_n_inf_and_tau(v)
    crossing_count = np.zeros(count, dtype=int)
    crossing_times = np.zeros((count, 3))
    times = [0.0]
    voltage_history = [v.copy()]
    h_history = [h.copy()]
    n_history = [n.copy()]
    t_ms = 0.0

    while np.any(crossing_count < 3) and t_ms < 2000.0:
        v_old = v.copy()
        t_old = t_ms
        m = _wb_m_inf(v)
        h_inf, tau_h = _wb_h_inf_and_tau(v)
        n_inf, tau_n = _wb_n_inf_and_tau(v)
        dv = (
            9.0 * n**4 * (-90.0 - v)
            + 35.0 * m**3 * h * (55.0 - v)
            + 0.1 * (-65.0 - v)
            + i_ext_ua
        )
        dh = (h_inf - h) / tau_h
        dn = (n_inf - n) / tau_n

        v_mid = v + 0.5 * dt_ms * dv
        h_mid = h + 0.5 * dt_ms * dh
        n_mid = n + 0.5 * dt_ms * dn
        m_mid = _wb_m_inf(v)
        h_inf_mid, tau_h_mid = _wb_h_inf_and_tau(v_mid)
        n_inf_mid, tau_n_mid = _wb_n_inf_and_tau(v_mid)
        dv_mid = (
            9.0 * n_mid**4 * (-90.0 - v_mid)
            + 35.0 * m_mid**3 * h_mid * (55.0 - v_mid)
            + 0.1 * (-65.0 - v_mid)
            + i_ext_ua
        )
        dh_mid = (h_inf_mid - h_mid) / tau_h_mid
        dn_mid = (n_inf_mid - n_mid) / tau_n_mid
        v = v + dt_ms * dv_mid
        h = h + dt_ms * dh_mid
        n = n + dt_ms * dn_mid
        t_ms += dt_ms

        crossed = np.flatnonzero((v_old >= -20.0) & (v < -20.0))
        for cell in crossed:
            if crossing_count[cell] < 3:
                fraction = (-20.0 - v_old[cell]) / (v[cell] - v_old[cell])
                crossing_times[cell, crossing_count[cell]] = t_old + fraction * dt_ms
                crossing_count[cell] += 1

        times.append(t_ms)
        voltage_history.append(v.copy())
        h_history.append(h.copy())
        n_history.append(n.copy())

    if np.any(crossing_count < 3):
        missing = np.flatnonzero(crossing_count < 3)
        raise RuntimeError(
            f"WB phase initialization did not find three spikes for cells {missing.tolist()}"
        )

    times = np.asarray(times)
    voltage_history = np.asarray(voltage_history)
    h_history = np.asarray(h_history)
    n_history = np.asarray(n_history)
    target_times = crossing_times[:, 1] + phase * (
        crossing_times[:, 2] - crossing_times[:, 1]
    )
    upper = np.searchsorted(times, target_times, side="right")
    upper = np.minimum(upper, len(times) - 1)
    lower = upper - 1
    fraction = (target_times - times[lower]) / (times[upper] - times[lower])
    cells = np.arange(count)

    def interpolate(history):
        return (
            history[lower, cells]
            + fraction * (history[upper, cells] - history[lower, cells])
        )

    return {
        "v": interpolate(voltage_history) * b2.mV,
        "h": interpolate(h_history),
        "n": interpolate(n_history),
        "q": np.zeros(count),
        "s": np.zeros(count),
    }


def build_ing_inputs(config, seed=None, num_i=None):
    config = dict(config)
    num_i = 100 if num_i is None else int(num_i)
    if num_i < 1:
        raise ValueError("num_i must be positive")
    p_ii = validate_probability("p_ii", config["p_ii"])
    p_gap = validate_probability("p_gap", config["p_gap"])
    if p_ii == 0.0 and config["g_hat_ii"] != 0.0:
        raise ValueError("nonzero g_hat_ii requires p_ii > 0")
    if p_gap == 0.0 and config["g_hat_gap"] != 0.0:
        raise ValueError("nonzero g_hat_gap requires p_gap > 0")

    rng = np.random.default_rng(config["seed"] if seed is None else seed)
    i_ext_ua = 1.5 * (
        1.0 + config["sigma_i"] * rng.standard_normal(num_i)
    )

    g_ii_ms = np.zeros((num_i, num_i))
    if config["g_hat_ii"] != 0.0:
        weight = config["g_hat_ii"] / (num_i * p_ii)
        if config["fixed_indegree"]:
            g_ii_ms.fill(weight)
            omit = round(num_i - p_ii * num_i)
            for post in range(num_i):
                drop = rng.choice(num_i, size=omit, replace=False)
                g_ii_ms[drop, post] = 0.0
        else:
            g_ii_ms = weight * (rng.random((num_i, num_i)) < p_ii)

    g_gap_ms = np.zeros((num_i, num_i))
    if config["g_hat_gap"] != 0.0 and num_i > 1:
        weight = config["g_hat_gap"] / (p_gap * (num_i - 1))
        for left in range(num_i - 1):
            for right in range(left + 1, num_i):
                if rng.random() < p_gap:
                    g_gap_ms[left, right] = weight
                    g_gap_ms[right, left] = weight

    if config["init_mode"] == "phase":
        initial_state = wb_phase_initial_state(
            i_ext_ua, rng.random(num_i)
        )
    else:
        initial_state = initial_wb_state(
            num_i, config["init_mode"], rng
        )
    return {
        "i_ext_ua": i_ext_ua,
        "g_ii_ms": g_ii_ms,
        "g_gap_ms": g_gap_ms,
        "initial_state": initial_state,
    }

In [ ]:
def simulate_ing_network(
    name,
    duration=500 * b2.ms,
    dt=0.01 * b2.ms,
    num_i=100,
    seed=None,
    record=False,
):
    if name not in ING_CONFIGS:
        raise ValueError(f"unknown ING configuration: {name}")
    config = dict(ING_CONFIGS[name])
    inputs = build_ing_inputs(config, seed=seed, num_i=num_i)

    b2.start_scope()
    b2.defaultclock.dt = dt
    cells = b2.NeuronGroup(
        num_i,
        WB_EQS,
        method="rk2",
        threshold="v > -20*mV",
        refractory="v > -20*mV",
    )
    cells.C = 1.0 * b2.ufarad
    cells.g_l = 0.1 * b2.msiemens
    cells.g_k = 9.0 * b2.msiemens
    cells.g_na = 35.0 * b2.msiemens
    cells.E_l = -65.0 * b2.mV
    cells.E_k = -90.0 * b2.mV
    cells.E_na = 55.0 * b2.mV
    cells.i_ext = inputs["i_ext_ua"] * b2.uA
    cells.tau_dq = solve_tau_dq(9.0, 0.5, 0.5) * b2.ms
    cells.tau_r = 0.5 * b2.ms
    cells.tau_d = 9.0 * b2.ms
    for variable, values in inputs["initial_state"].items():
        setattr(cells, variable, values)

    pre, post = np.nonzero(inputs["g_ii_ms"])
    chemical = b2.Synapses(
        cells,
        cells,
        model="g : siemens\nI_chem_post = g*s_pre*(-75*mV-v_post) : amp (summed)",
    )
    if pre.size:
        chemical.connect(i=pre, j=post)
        chemical.g = inputs["g_ii_ms"][pre, post] * b2.msiemens

    post, pre = np.nonzero(inputs["g_gap_ms"])
    gap = b2.Synapses(
        cells,
        cells,
        model="g : siemens\nI_gap_post = g*(v_pre-v_post) : amp (summed)",
    )
    if pre.size:
        gap.connect(i=pre, j=post)
        gap.g = inputs["g_gap_ms"][post, pre] * b2.msiemens

    spike_monitor = b2.SpikeMonitor(cells)
    state_monitor = (
        b2.StateMonitor(cells, ("v", "q", "s"), record=True)
        if record
        else None
    )
    b2.run(duration)

    result = {
        "t_i_ms": np.asarray(spike_monitor.t / b2.ms),
        "i_i": np.asarray(spike_monitor.i, dtype=int),
        "duration_ms": float(duration / b2.ms),
        "num_i": int(num_i),
        "config": config,
    }
    if state_monitor is not None:
        result["state"] = {
            "t_ms": np.asarray(state_monitor.t / b2.ms),
            "v_mv": np.asarray(state_monitor.v / b2.mV),
            "q": np.asarray(state_monitor.q),
            "s": np.asarray(state_monitor.s),
        }
    return result


def plot_ing_raster(result, ax=None, closeup=False):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))
    ax.plot(result["t_i_ms"], result["i_i"], ".k", markersize=2)
    if closeup:
        ax.set_xlim(result["duration_ms"] - 100.0, result["duration_ms"])
        ax.set_ylim(0, 21)
    else:
        ax.set_xlim(0, result["duration_ms"])
        ax.set_ylim(0, result["num_i"] + 1)
    ax.set_xlabel("t [ms]")
    return ax

In [ ]:
if __name__ == "__main__":
    ing_1 = simulate_ing_network("ING_1")
    plot_ing_raster(ing_1)

In [ ]:
if __name__ == "__main__":
    ing_2 = simulate_ing_network("ING_2")
    plot_ing_raster(ing_2)

In [ ]:
if __name__ == "__main__":
    ing_3 = simulate_ing_network("ING_3")
    plot_ing_raster(ing_3)

In [ ]:
if __name__ == "__main__":
    ing_4 = simulate_ing_network("ING_4")
    plot_ing_raster(ing_4)

In [ ]:
if __name__ == "__main__":
    ing_5 = simulate_ing_network("ING_5")
    plot_ing_raster(ing_5)

In [ ]:
if __name__ == "__main__":
    ing_6 = simulate_ing_network("ING_6")
    plot_ing_raster(ing_6)

In [ ]:
if __name__ == "__main__":
    ing_7 = simulate_ing_network("ING_7")
    plot_ing_raster(ing_7)

In [ ]:
if __name__ == "__main__":
    ing_8 = simulate_ing_network("ING_8")
    plot_ing_raster(ing_8, closeup=True)

In [ ]:
if __name__ == "__main__":
    ing_9 = simulate_ing_network("ING_9")
    plot_ing_raster(ing_9, closeup=True)

In [ ]:
if __name__ == "__main__":
    ing_10 = simulate_ing_network("ING_10")
    plot_ing_raster(ing_10, closeup=True)

In [ ]:
EI_WB_EQS = (
    WB_EQS
    .replace("i_ext + I_chem + I_gap", "i_ext + I_from_e + I_from_i + I_gap")
    .replace("I_chem : amp", "I_from_e : amp\nI_from_i : amp")
)

RTM_EQS = """
dv/dt = (g_l*(E_l-v) + g_k*n**4*(E_k-v)
         + g_na*m_inf**3*h*(E_na-v) + i_ext + I_from_e + I_from_i)/C : volt
dh/dt = (h_inf-h)/tau_h : 1
dn/dt = (n_inf-n)/tau_n : 1
dq/dt = 0.5*(1+tanh(v/(10*mV)))*(1-q)/(0.1*ms) - q/tau_dq : 1
ds/dt = q*(1-s)/tau_r - s/tau_d : 1
m_inf = alpha_m/(alpha_m+beta_m) : 1
h_inf = alpha_h/(alpha_h+beta_h) : 1
n_inf = alpha_n/(alpha_n+beta_n) : 1
alpha_m = 0.32/mV*(v+54*mV)/(1-exp(-(v+54*mV)/(4*mV)))/ms : Hz
beta_m = 0.28/mV*(v+27*mV)/(exp((v+27*mV)/(5*mV))-1)/ms : Hz
alpha_h = 0.128*exp(-(v+50*mV)/(18*mV))/ms : Hz
beta_h = 4/(1+exp(-(v+27*mV)/(5*mV)))/ms : Hz
alpha_n = 0.032/mV*(v+52*mV)/(1-exp(-(v+52*mV)/(5*mV)))/ms : Hz
beta_n = 0.5*exp(-(v+57*mV)/(40*mV))/ms : Hz
tau_h = 1/(alpha_h+beta_h) : second
tau_n = 1/(alpha_n+beta_n) : second
i_ext : amp
I_from_e : amp
I_from_i : amp
tau_dq : second
tau_r : second
tau_d : second
C : farad (constant)
g_l : siemens (constant)
g_k : siemens (constant)
g_na : siemens (constant)
E_l : volt (constant)
E_k : volt (constant)
E_na : volt (constant)
"""


ENTRAINMENT_CONFIGS = {
    "ING_ENTRAINING_E_CELLS": dict(
        num_e=400, num_i=100, sigma_e=0.10, sigma_i=0.05,
        mean_e=1.5, mean_i=1.5,
        g_hat_ee=0.0, g_hat_ei=0.0, g_hat_ie=0.50, g_hat_ii=0.50,
        p_ee=1.0, p_ei=0.5, p_ie=0.5, p_ii=0.5,
        g_hat_gap=0.1, p_gap=0.05,
    ),
    "ING_ENTRAINING_E_CELLS_2": dict(
        num_e=400, num_i=100, sigma_e=0.00, sigma_i=0.00,
        mean_e=1.9, mean_i=1.5,
        g_hat_ee=0.0, g_hat_ei=0.0, g_hat_ie=0.50, g_hat_ii=0.50,
        p_ee=1.0, p_ei=1.0, p_ie=1.0, p_ii=1.0,
        g_hat_gap=0.1, p_gap=0.05,
    ),
}


def _rtm_m_inf(v):
    alpha_m = 0.32 * (v + 54.0) / (1.0 - np.exp(-(v + 54.0) / 4.0))
    beta_m = 0.28 * (v + 27.0) / (np.exp((v + 27.0) / 5.0) - 1.0)
    return alpha_m / (alpha_m + beta_m)


def _rtm_h_inf_and_tau(v):
    alpha_h = 0.128 * np.exp(-(v + 50.0) / 18.0)
    beta_h = 4.0 / (1.0 + np.exp(-(v + 27.0) / 5.0))
    rate = alpha_h + beta_h
    return alpha_h / rate, 1.0 / rate


def _rtm_n_inf_and_tau(v):
    alpha_n = 0.032 * (v + 52.0) / (1.0 - np.exp(-(v + 52.0) / 5.0))
    beta_n = 0.5 * np.exp(-(v + 57.0) / 40.0)
    rate = alpha_n + beta_n
    return alpha_n / rate, 1.0 / rate


def rtm_phase_initial_state(i_ext_ua, phase, dt_ms=0.01):
    i_ext_ua = np.asarray(i_ext_ua, dtype=float)
    phase = np.asarray(phase, dtype=float)
    if i_ext_ua.ndim != 1 or phase.shape != i_ext_ua.shape:
        raise ValueError("i_ext_ua and phase must be equal-length one-dimensional arrays")
    if i_ext_ua.size == 0:
        raise ValueError("i_ext_ua and phase must not be empty")
    if np.any(~np.isfinite(i_ext_ua)) or np.any(~np.isfinite(phase)):
        raise ValueError("i_ext_ua and phase must contain finite values")
    if np.any((phase < 0.0) | (phase > 1.0)):
        raise ValueError("phase must be in [0, 1]")
    dt_ms = float(dt_ms)
    if not np.isfinite(dt_ms) or dt_ms <= 0.0:
        raise ValueError("dt_ms must be positive")

    count = i_ext_ua.size
    v = np.full(count, -70.0)
    h, _ = _rtm_h_inf_and_tau(v)
    n, _ = _rtm_n_inf_and_tau(v)
    crossing_count = np.zeros(count, dtype=int)
    crossing_times = np.zeros((count, 3))
    target_times = np.full(count, np.inf)
    done = np.zeros(count, dtype=bool)
    output_v = np.zeros(count)
    output_h = np.zeros(count)
    output_n = np.zeros(count)
    t_ms = 0.0

    while np.any(~done) and t_ms < 2000.0:
        v_old = v.copy()
        h_old = h.copy()
        n_old = n.copy()
        t_old = t_ms
        m = _rtm_m_inf(v)
        h_inf, tau_h = _rtm_h_inf_and_tau(v)
        n_inf, tau_n = _rtm_n_inf_and_tau(v)
        dv = (
            80.0 * n**4 * (-100.0 - v)
            + 100.0 * m**3 * h * (50.0 - v)
            + 0.1 * (-67.0 - v)
            + i_ext_ua
        )
        dh = (h_inf - h) / tau_h
        dn = (n_inf - n) / tau_n

        v_mid = v + 0.5 * dt_ms * dv
        h_mid = h + 0.5 * dt_ms * dh
        n_mid = n + 0.5 * dt_ms * dn
        m_mid = _rtm_m_inf(v)
        h_inf_mid, tau_h_mid = _rtm_h_inf_and_tau(v_mid)
        n_inf_mid, tau_n_mid = _rtm_n_inf_and_tau(v_mid)
        dv_mid = (
            80.0 * n_mid**4 * (-100.0 - v_mid)
            + 100.0 * m_mid**3 * h_mid * (50.0 - v_mid)
            + 0.1 * (-67.0 - v_mid)
            + i_ext_ua
        )
        dh_mid = (h_inf_mid - h_mid) / tau_h_mid
        dn_mid = (n_inf_mid - n_mid) / tau_n_mid
        v = v + dt_ms * dv_mid
        h = h + dt_ms * dh_mid
        n = n + dt_ms * dn_mid
        t_ms += dt_ms

        crossed = np.flatnonzero((v_old >= -20.0) & (v < -20.0))
        for cell in crossed:
            if crossing_count[cell] < 3:
                fraction = (-20.0 - v_old[cell]) / (v[cell] - v_old[cell])
                crossing_times[cell, crossing_count[cell]] = t_old + fraction * dt_ms
                crossing_count[cell] += 1
                if crossing_count[cell] == 3:
                    period = crossing_times[cell, 2] - crossing_times[cell, 1]
                    target_times[cell] = crossing_times[cell, 2] + phase[cell] * period

        reached = np.flatnonzero((~done) & (t_old <= target_times) & (target_times < t_ms))
        if reached.size:
            fraction = (target_times[reached] - t_old) / dt_ms
            output_v[reached] = v_old[reached] + fraction * (v[reached] - v_old[reached])
            output_h[reached] = h_old[reached] + fraction * (h[reached] - h_old[reached])
            output_n[reached] = n_old[reached] + fraction * (n[reached] - n_old[reached])
            done[reached] = True

    if np.any(~done):
        missing = np.flatnonzero(~done)
        raise RuntimeError(
            f"RTM phase initialization did not finish for cells {missing.tolist()}"
        )
    return {
        "v": output_v * b2.mV,
        "h": output_h,
        "n": output_n,
        "q": np.zeros(count),
        "s": np.zeros(count),
    }


def wb_entrainment_phase_initial_state(i_ext_ua, phase, dt_ms=0.01):
    i_ext_ua = np.asarray(i_ext_ua, dtype=float)
    phase = np.asarray(phase, dtype=float)
    if i_ext_ua.ndim != 1 or phase.shape != i_ext_ua.shape:
        raise ValueError("i_ext_ua and phase must be equal-length one-dimensional arrays")
    if i_ext_ua.size == 0:
        raise ValueError("i_ext_ua and phase must not be empty")
    if np.any(~np.isfinite(i_ext_ua)) or np.any(~np.isfinite(phase)):
        raise ValueError("i_ext_ua and phase must contain finite values")
    if np.any((phase < 0.0) | (phase > 1.0)):
        raise ValueError("phase must be in [0, 1]")
    dt_ms = float(dt_ms)
    if not np.isfinite(dt_ms) or dt_ms <= 0.0:
        raise ValueError("dt_ms must be positive")

    count = i_ext_ua.size
    v = np.full(count, -70.0)
    h, _ = _wb_h_inf_and_tau(v)
    n, _ = _wb_n_inf_and_tau(v)
    crossing_count = np.zeros(count, dtype=int)
    crossing_times = np.zeros((count, 3))
    target_times = np.full(count, np.inf)
    done = np.zeros(count, dtype=bool)
    output_v = np.zeros(count)
    output_h = np.zeros(count)
    output_n = np.zeros(count)
    t_ms = 0.0

    while np.any(~done) and t_ms < 2000.0:
        v_old = v.copy()
        h_old = h.copy()
        n_old = n.copy()
        t_old = t_ms
        m = _wb_m_inf(v)
        h_inf, tau_h = _wb_h_inf_and_tau(v)
        n_inf, tau_n = _wb_n_inf_and_tau(v)
        dv = (
            9.0 * n**4 * (-90.0 - v)
            + 35.0 * m**3 * h * (55.0 - v)
            + 0.1 * (-65.0 - v)
            + i_ext_ua
        )
        dh = (h_inf - h) / tau_h
        dn = (n_inf - n) / tau_n

        v_mid = v + 0.5 * dt_ms * dv
        h_mid = h + 0.5 * dt_ms * dh
        n_mid = n + 0.5 * dt_ms * dn
        m_mid = _wb_m_inf(v)
        h_inf_mid, tau_h_mid = _wb_h_inf_and_tau(v_mid)
        n_inf_mid, tau_n_mid = _wb_n_inf_and_tau(v_mid)
        dv_mid = (
            9.0 * n_mid**4 * (-90.0 - v_mid)
            + 35.0 * m_mid**3 * h_mid * (55.0 - v_mid)
            + 0.1 * (-65.0 - v_mid)
            + i_ext_ua
        )
        dh_mid = (h_inf_mid - h_mid) / tau_h_mid
        dn_mid = (n_inf_mid - n_mid) / tau_n_mid
        v = v + dt_ms * dv_mid
        h = h + dt_ms * dh_mid
        n = n + dt_ms * dn_mid
        t_ms += dt_ms

        crossed = np.flatnonzero((v_old >= -20.0) & (v < -20.0))
        for cell in crossed:
            if crossing_count[cell] < 3:
                fraction = (-20.0 - v_old[cell]) / (v[cell] - v_old[cell])
                crossing_times[cell, crossing_count[cell]] = t_old + fraction * dt_ms
                crossing_count[cell] += 1
                if crossing_count[cell] == 3:
                    period = crossing_times[cell, 2] - crossing_times[cell, 1]
                    target_times[cell] = crossing_times[cell, 2] + phase[cell] * period

        reached = np.flatnonzero((~done) & (t_old <= target_times) & (target_times < t_ms))
        if reached.size:
            fraction = (target_times[reached] - t_old) / dt_ms
            output_v[reached] = v_old[reached] + fraction * (v[reached] - v_old[reached])
            output_h[reached] = h_old[reached] + fraction * (h[reached] - h_old[reached])
            output_n[reached] = n_old[reached] + fraction * (n[reached] - n_old[reached])
            done[reached] = True

    if np.any(~done):
        missing = np.flatnonzero(~done)
        raise RuntimeError(
            f"WB entrainment phase initialization did not finish for cells {missing.tolist()}"
        )
    return {
        "v": output_v * b2.mV,
        "h": output_h,
        "n": output_n,
        "q": np.zeros(count),
        "s": np.zeros(count),
    }


def _entrainment_chemical_matrix(rng, source_count, target_count, g_hat, p):
    p = validate_probability("chemical connection probability", p)
    if p == 0.0:
        if g_hat != 0.0:
            raise ValueError("nonzero chemical strength requires positive probability")
        return np.zeros((source_count, target_count))
    return (
        g_hat
        * (rng.random((source_count, target_count)) < p)
        / (source_count * p)
    )


def build_entrainment_inputs(config, seed, num_e, num_i):
    rng = np.random.default_rng(seed)
    i_ext_e_ua = config["mean_e"] * (
        1.0 + config["sigma_e"] * rng.standard_normal(num_e)
    )
    i_ext_i_ua = config["mean_i"] * (
        1.0 + config["sigma_i"] * rng.standard_normal(num_i)
    )
    g_ee_ms = _entrainment_chemical_matrix(
        rng, num_e, num_e, config["g_hat_ee"], config["p_ee"]
    )
    g_ei_ms = _entrainment_chemical_matrix(
        rng, num_e, num_i, config["g_hat_ei"], config["p_ei"]
    )
    g_ie_ms = _entrainment_chemical_matrix(
        rng, num_i, num_e, config["g_hat_ie"], config["p_ie"]
    )
    g_ii_ms = _entrainment_chemical_matrix(
        rng, num_i, num_i, config["g_hat_ii"], config["p_ii"]
    )

    p_gap = validate_probability("p_gap", config["p_gap"])
    if p_gap == 0.0 and config["g_hat_gap"] != 0.0:
        raise ValueError("nonzero g_hat_gap requires p_gap > 0")
    g_gap_ms = np.zeros((num_i, num_i))
    if config["g_hat_gap"] != 0.0 and num_i > 1:
        weight = config["g_hat_gap"] / (p_gap * (num_i - 1))
        for left in range(num_i - 1):
            for right in range(left + 1, num_i):
                if rng.random() < p_gap:
                    g_gap_ms[left, right] = weight
                    g_gap_ms[right, left] = weight

    initial_e = rtm_phase_initial_state(i_ext_e_ua, rng.random(num_e))
    initial_i = wb_entrainment_phase_initial_state(i_ext_i_ua, rng.random(num_i))
    return {
        "i_ext_e_ua": i_ext_e_ua,
        "i_ext_i_ua": i_ext_i_ua,
        "g_ee_ms": g_ee_ms,
        "g_ei_ms": g_ei_ms,
        "g_ie_ms": g_ie_ms,
        "g_ii_ms": g_ii_ms,
        "g_gap_ms": g_gap_ms,
        "initial_e": initial_e,
        "initial_i": initial_i,
    }

In [ ]:
def add_continuous_synapses(source, target, weights_ms, reversal, target_variable):
    pre, post = np.nonzero(weights_ms)
    model = (
        "g : siemens\n"
        "E_rev : volt (constant)\n"
        f"{target_variable}_post = g*s_pre*(E_rev-v_post) : amp (summed)"
    )
    synapses = b2.Synapses(source, target, model=model)
    if pre.size:
        synapses.connect(i=pre, j=post)
        synapses.g = weights_ms[pre, post] * b2.msiemens
    else:
        synapses.connect(condition="False")
    synapses.E_rev = reversal
    return synapses


def simulate_ing_entrainment(
    variant,
    e_drive=None,
    duration=500 * b2.ms,
    dt=0.01 * b2.ms,
    num_e=400,
    num_i=100,
    seed=63806,
):
    if variant not in ENTRAINMENT_CONFIGS:
        raise ValueError(f"unknown entrainment variant: {variant}")
    num_e = int(num_e)
    num_i = int(num_i)
    if num_e < 1 or num_i < 1:
        raise ValueError("num_e and num_i must be positive")
    config = dict(ENTRAINMENT_CONFIGS[variant])
    config["num_e"] = num_e
    config["num_i"] = num_i
    if e_drive is not None:
        e_drive = float(e_drive)
        if variant != "ING_ENTRAINING_E_CELLS_2":
            raise ValueError("e_drive is only selectable for ING_ENTRAINING_E_CELLS_2")
        if e_drive not in (1.9, 2.0, 2.1):
            raise ValueError("e_drive must be one of 1.9, 2.0, or 2.1")
        config["mean_e"] = e_drive
    inputs = build_entrainment_inputs(config, seed, num_e, num_i)

    b2.start_scope()
    b2.defaultclock.dt = dt
    excitatory = b2.NeuronGroup(
        num_e, RTM_EQS, method="rk2",
        threshold="v > -20*mV", refractory="v > -20*mV",
    )
    excitatory.C = 1.0 * b2.ufarad
    excitatory.g_l = 0.1 * b2.msiemens
    excitatory.g_k = 80.0 * b2.msiemens
    excitatory.g_na = 100.0 * b2.msiemens
    excitatory.E_l = -67.0 * b2.mV
    excitatory.E_k = -100.0 * b2.mV
    excitatory.E_na = 50.0 * b2.mV
    excitatory.i_ext = inputs["i_ext_e_ua"] * b2.uA
    excitatory.tau_dq = solve_tau_dq(3.0, 0.5, 0.5) * b2.ms
    excitatory.tau_r = 0.5 * b2.ms
    excitatory.tau_d = 3.0 * b2.ms
    for variable, values in inputs["initial_e"].items():
        setattr(excitatory, variable, values)

    inhibitory = b2.NeuronGroup(
        num_i, EI_WB_EQS, method="rk2",
        threshold="v > -20*mV", refractory="v > -20*mV",
    )
    inhibitory.C = 1.0 * b2.ufarad
    inhibitory.g_l = 0.1 * b2.msiemens
    inhibitory.g_k = 9.0 * b2.msiemens
    inhibitory.g_na = 35.0 * b2.msiemens
    inhibitory.E_l = -65.0 * b2.mV
    inhibitory.E_k = -90.0 * b2.mV
    inhibitory.E_na = 55.0 * b2.mV
    inhibitory.i_ext = inputs["i_ext_i_ua"] * b2.uA
    inhibitory.tau_dq = solve_tau_dq(9.0, 0.5, 0.5) * b2.ms
    inhibitory.tau_r = 0.5 * b2.ms
    inhibitory.tau_d = 9.0 * b2.ms
    for variable, values in inputs["initial_i"].items():
        setattr(inhibitory, variable, values)

    e_to_e = add_continuous_synapses(
        excitatory, excitatory, inputs["g_ee_ms"], 0.0 * b2.mV, "I_from_e"
    )
    e_to_i = add_continuous_synapses(
        excitatory, inhibitory, inputs["g_ei_ms"], 0.0 * b2.mV, "I_from_e"
    )
    i_to_e = add_continuous_synapses(
        inhibitory, excitatory, inputs["g_ie_ms"], -75.0 * b2.mV, "I_from_i"
    )
    i_to_i = add_continuous_synapses(
        inhibitory, inhibitory, inputs["g_ii_ms"], -75.0 * b2.mV, "I_from_i"
    )

    gap_post, gap_pre = np.nonzero(inputs["g_gap_ms"])
    gap = b2.Synapses(
        inhibitory,
        inhibitory,
        model="g : siemens\nI_gap_post = g*(v_pre-v_post) : amp (summed)",
    )
    if gap_pre.size:
        gap.connect(i=gap_pre, j=gap_post)
        gap.g = inputs["g_gap_ms"][gap_post, gap_pre] * b2.msiemens
    else:
        gap.connect(condition="False")

    spike_e = b2.SpikeMonitor(excitatory)
    spike_i = b2.SpikeMonitor(inhibitory)
    b2.run(duration)
    return {
        "t_e_ms": np.asarray(spike_e.t / b2.ms),
        "i_e": np.asarray(spike_e.i, dtype=int),
        "t_i_ms": np.asarray(spike_i.t / b2.ms),
        "i_i": np.asarray(spike_i.i, dtype=int),
        "duration_ms": float(duration / b2.ms),
        "num_e": num_e,
        "num_i": num_i,
        "config": config,
    }


def plot_ing_entrainment(result, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 4))
    ax.plot(result["t_i_ms"], result["i_i"], ".b", markersize=2)
    ax.plot(
        result["t_e_ms"], result["i_e"] + result["num_i"],
        ".r", markersize=2,
    )
    boundary = result["num_i"] + 0.5
    ax.plot([0, result["duration_ms"]], [boundary, boundary], "--k", linewidth=1)
    ax.set_yticks([result["num_i"], result["num_i"] + result["num_e"]])
    ax.axis([0, result["duration_ms"], 0, result["num_e"] + result["num_i"] + 1])
    ax.set_xlabel("t [ms]")
    return ax


def plot_ing_entrainment_sweep(results, axes=None):
    if len(results) != 3:
        raise ValueError("the entrainment sweep requires three results")
    if axes is None:
        _, axes = plt.subplots(3, 1, figsize=(8, 9))
    axes = np.asarray(axes).reshape(-1)
    if axes.size != 3:
        raise ValueError("the entrainment sweep requires three axes")
    for ax, result in zip(axes, results):
        plot_ing_entrainment(result, ax=ax)
        ax.set_title(rf"$\overline{{I}}_E={result['config']['mean_e']:g}$")
        ax.set_xlabel("")
    axes[-1].set_xlabel("t [ms]")
    return axes

In [ ]:
if __name__ == "__main__":
    ing_entraining_e_cells = simulate_ing_entrainment("ING_ENTRAINING_E_CELLS")
    plot_ing_entrainment(ing_entraining_e_cells)

In [ ]:
if __name__ == "__main__":
    entrainment_sweep = [
        simulate_ing_entrainment("ING_ENTRAINING_E_CELLS_2", e_drive=drive)
        for drive in (1.9, 2.0, 2.1)
    ]
    plot_ing_entrainment_sweep(entrainment_sweep)